# OmniVoice Studio · Bản sạch đã gộp bản vá

Notebook này đã tích hợp **cấu hình ngrok v2** và **đăng nhập Web UI bằng phiên riêng/mã dùng một lần**.
Không cần các ô kiểm tra khóa, đọc log ngrok hoặc file vá đăng nhập thêm từ cuộc trao đổi trước.

## Lần sau khởi động lại: chạy những ô nào?

Khi Colab tạo máy ảo mới hoặc khởi động lại runtime, chọn GPU rồi chạy theo thứ tự:

**1a → 1b → 2 → 3 → 5a → 5b → 6 → 7a → 7b**.

| Ô | Chức năng |
|---|---|
| 1a, 1b | Kiểm tra GPU, kết nối đúng tài khoản Drive |
| 2 | Đọc `OMNIVOICE_API_KEY` và `NGROK_AUTHTOKEN` |
| 3 | Cài lại thư viện trong máy ảo mới |
| 5a, 5b | Ghi bản server/UI đã sửa, nạp model và các giọng từ Drive |
| 6 | Thử tạo WAV và chuẩn bị biến dùng ở phần 7 |
| 7a, 7b | Ghi tunnel đã sửa, mở ngrok và tạo liên kết đăng nhập Web UI |

Phần 4 chỉ hướng dẫn thêm giọng trên web. Hai ô **Xem log** và **Dừng** là tùy chọn.
Có thể dùng **Chạy tất cả**; giữ `STOP = False` ở ô cuối để API tiếp tục chạy.

**Khi 7b xong:** bấm **Mở Web UI · đăng nhập trực tiếp** trong 5 phút. Nếu cần liên kết mới,
chạy `open_webui_session()` trong một ô mã mới. Lấy Base URL của phiên đang chạy cho ReviewTrans.

Model nằm ở `MyDrive/OmniVoice/models`, giọng ở `MyDrive/OmniVoice/voices`, thiết lập ở
`MyDrive/OmniVoice/settings.json`; không cần upload lại giọng đã lưu. Thư viện Python vẫn cài
lại cho máy ảo mới. Hugging Face chỉ tải thêm file model thiếu hoặc thay đổi.

Nếu chỉ tải lại trang Web UI và runtime/server còn chạy, bạn có thể mở lại giao diện hiện tại;
không cần chạy lại các ô khởi động. Nếu chỉ cần liên kết đăng nhập mới, dùng lệnh ở trên.

---

## Hướng dẫn cài và sử dụng

Notebook tự chứa Web UI tiếng Việt và mã API; không cần tải file phụ. Giao thức được đối chiếu với
`OpenAISpeechTTS` trong bản ReviewTrans bạn cung cấp ngày 12/11/2026.

**Mới:** thêm nhiều giọng clone trên web, nghe thử, chỉnh thông số và lưu thiết lập cho API.
Web UI và ReviewTrans dùng chung một model; chỉ mở một tiến trình server.

**Luồng:** ReviewTrans trên PC → HTTPS ngrok → FastAPI trên Colab → GPU OmniVoice → WAV.

1. Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU mạnh hơn).
2. Tạo tài khoản ngrok và lấy [authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
3. Chạy lần lượt các ô 1–7. Ô 7 in liên kết Web UI và bảng thông tin để điền vào ReviewTrans.

**Đã bật lưu model vào Google Drive.** Ở phần 1, cho phép Colab kết nối Drive.
Lần đầu chạy phần 5 sẽ tải trọng số vào `MyDrive/OmniVoice/models`; những lần sau
dùng lại file đã tải. Nếu model trên Hub có phiên bản mới, có thể tải thêm phần thay đổi.
Thư viện Python vẫn được cài lại khi Colab cấp máy ảo mới; không lưu môi trường Python lên Drive.

Có thể đặt `NGROK_AUTHTOKEN` và `OMNIVOICE_API_KEY` trong **Secrets** (biểu tượng chìa khóa,
bật Notebook access). Nếu chưa đặt, notebook hỏi bằng ô nhập ẩn. API key là chuỗi tự chọn
tối thiểu 24 ký tự; dùng lại chính chuỗi đó trên PC. **Không gửi token/key vào chat.**

T4 là cấu hình khởi đầu đề xuất, không phải cam kết tốc độ/bộ nhớ. Notebook dùng FP16,
16 bước, một yêu cầu GPU tại một thời điểm; không nạp Whisper. Cài riêng môi trường
để tránh xung đột với thư viện có sẵn của Colab. Dự trù khoảng 20 GB ổ đĩa trống.

Đây là phiên thử nghiệm tương tác. Colab có thể thu hồi runtime; URL ngrok chỉ hoạt động
khi tiến trình và runtime còn chạy. Không dùng notebook này làm dịch vụ thường trực.
Colab miễn phí có thể kết thúc phiên khi cách dùng chủ yếu thông qua giao diện ngoài notebook.
Xem [FAQ chính thức](https://research.google.com/colaboratory/faq.html).


## 1. Kiểm tra GPU và kết nối Google Drive


In [ ]:
# 1a · Kiểm tra GPU

import os, sys, subprocess, time, json, urllib.request, urllib.error
from pathlib import Path
from getpass import getpass

if "google.colab" not in sys.modules:
    import google.colab
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Bản này dùng Python 3.10–3.13. Chọn runtime Colab tương thích PyTorch 2.8.")
subprocess.run(["nvidia-smi"], check=True)
APP_DIR = Path("/content/omnivoice-reviewtrans")
APP_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])


### Lưu model lâu dài trên Drive

Giữ `SAVE_MODELS_TO_DRIVE = True`. Khi Colab hỏi quyền truy cập, chọn tài khoản Drive
bạn muốn lưu model. Thư mục được tạo tự động; lần đầu tải model diễn ra ở phần 5.

Trọng số/cache model, giọng thêm qua Web UI và thiết lập đã lưu nằm trên Drive. API key, ngrok authtoken và Hugging Face token
không được ghi vào thư mục đó. Model được đọc từ Drive nên bước nạp có thể chậm hơn ổ đĩa Colab.
Nếu Drive không kết nối được, ô sẽ báo lỗi để bạn xử lý trước khi tải model.


In [ ]:
# 1b · Kết nối Google Drive và chọn thư mục lưu

SAVE_MODELS_TO_DRIVE = True
if SAVE_MODELS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/OmniVoice")
    MODEL_CACHE = DRIVE_ROOT / "models"
else:
    MODEL_CACHE = APP_DIR / "temporary-models"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
print("Thư mục model:", MODEL_CACHE)
existing_repos = [p.name for p in MODEL_CACHE.iterdir() if p.is_dir() and p.name.startswith("models--")]
if existing_repos:
    print("Đã có cache model:", ", ".join(existing_repos))
    print("Phần 5 sẽ dùng lại file hợp lệ; chỉ tải file thiếu hoặc thay đổi.")
else:
    print("Chưa có cache. Phần 5 sẽ tải và lưu model lần đầu vào thư mục trên.")

STUDIO_DIR = DRIVE_ROOT if SAVE_MODELS_TO_DRIVE else APP_DIR / "studio"
STUDIO_DIR.mkdir(parents=True, exist_ok=True)
print("Giọng và thiết lập sẽ lưu trong:", STUDIO_DIR)


## 2. Nhập cấu hình và khóa

`OMNIVOICE_API_KEY` là khóa **bạn tự đặt cho API Colab**, không phải key OpenAI/Hugging Face
và cũng không phải authtoken ngrok. Mặc định ngôn ngữ `vi` cho phụ đề tiếng Việt.
Bắt đầu với **2 luồng** trong ReviewTrans và MAX_BATCH_SIZE = 2 ở ô 5a. Chưa cần sửa các tùy chọn khác.


In [ ]:
# 2 · Đọc khóa từ Colab Secrets

from google.colab import userdata

def read_secret(name, prompt):
    try:
        value = userdata.get(name)
    except Exception:
        value = ""
    return (value or getpass(prompt)).strip()

API_KEY = read_secret("OMNIVOICE_API_KEY", "Tự đặt API key (ít nhất 24 ký tự): ")
if len(API_KEY) < 24:
    raise ValueError("API key phải có ít nhất 24 ký tự. Chạy lại ô này.")
NGROK_TOKEN = read_secret("NGROK_AUTHTOKEN", "Dán ngrok authtoken: ")
if not NGROK_TOKEN:
    raise ValueError("Cần ngrok authtoken.")
LANGUAGE = "vi"
NUM_STEP = 16  # Đổi thành 32 nếu muốn thử chất lượng cao hơn, chậm hơn.
REFERENCE_PATH = ""
REFERENCE_TEXT = ""
print("Đã nhận cấu hình. Không in các khóa ra màn hình.")


## 3. Cài môi trường riêng

Lần đầu có thể mất nhiều phút để tải PyTorch và các phụ thuộc. Không cần cài CUDA toolkit
hay FlashAttention thủ công. PyTorch/torchaudio dùng 2.8.0 CUDA 12.8 theo ví dụ chính thức;
OmniVoice 0.2.1 và Transformers 5.3.0 được ghim phiên bản. Các phụ thuộc còn lại do pip giải quyết.


In [ ]:
# 3 · Cài thư viện cho máy ảo hiện tại

VENV = Path("/content/omnivoice-reviewtrans-venv")
PY = str(VENV / "bin/python")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
if not Path(PY).is_file():
    subprocess.run([sys.executable, "-m", "virtualenv", "--no-periodic-update", str(VENV)], check=True)
subprocess.run([PY, "-m", "pip", "install", "--upgrade", "pip"], check=True)
constraints = APP_DIR / "constraints.txt"
constraints.write_text("torch==2.8.0+cu128\ntorchaudio==2.8.0+cu128\ntransformers==5.3.0\n")
subprocess.run([
    PY, "-m", "pip", "install", "torch==2.8.0+cu128", "torchaudio==2.8.0+cu128",
    "--index-url", "https://download.pytorch.org/whl/cu128",
], check=True)
subprocess.run([
    PY, "-m", "pip", "install", "-c", str(constraints), "omnivoice==0.2.1",
    "transformers==5.3.0", "fastapi>=0.115,<1", "uvicorn>=0.30,<1",
    "pyngrok>=7.2,<10", "requests>=2.32,<3", "python-multipart>=0.0.20,<1",
], check=True)
subprocess.run([PY, "-m", "pip", "check"], check=True)
subprocess.run([PY, "-c", "import torch, torchaudio; from omnivoice import OmniVoice; "
    "assert torch.cuda.is_available(), 'CUDA chưa sẵn sàng; xem nvidia-smi/driver'; "
    "print('GPU:', torch.cuda.get_device_name(0)); "
    "x=torch.ones(8,device='cuda'); print('CUDA compute:', (x*x).sum().item())"], check=True)
with (APP_DIR / "installed-versions.txt").open("w") as f:
    subprocess.run([PY, "-m", "pip", "freeze"], stdout=f, check=True)
print("Cài đặt và phép tính GPU đã qua. Môi trường nằm trong /content, mất khi runtime bị xóa.")


## 4. Thêm giọng trên Web UI sau khi server chạy

Bản này không cần upload giọng bằng ô mã. Sau phần 7, mở Web UI, nhập cùng
API key ở phần 2, rồi dùng **Thêm giọng tham chiếu**:

1. Chọn mã giọng riêng, ví dụ `giong_nu_01`, và tên hiển thị.
2. Tải một file WAV/FLAC, tốt nhất 3–10 giây (chấp nhận 1–15 giây, tối đa 10 MB).
3. Nhập đúng lời nói trong mẫu và bấm **Lưu & thêm giọng clone**.

Giọng được thêm ngay vào `/v1/voices` và lưu trong `MyDrive/OmniVoice/voices`.
Mỗi lần khởi động, server nạp lại các mẫu này; không cần upload lại hay khởi động
lại server khi thêm giọng mới. ReviewTrans có thể cache giọng 10 phút: khởi động lại
ứng dụng nếu chưa thấy giọng mới. Chỉ dùng mẫu bạn có quyền sử dụng.


## 5. Tạo Web UI + API và nạp model

Ô này ghi mã server rồi khởi động đúng **một worker**. Lần đầu model tải từ Hugging Face,
có thể cần nhiều phút. Khi thấy `API sẵn sàng`, chuyển sang ô 6. Nếu lỗi, ô sẽ hiện phần
cuối log; không bỏ qua lỗi rồi mở tunnel.


In [ ]:
# 5a · Ghi server và Web UI đã sửa đăng nhập
MAX_BATCH_SIZE = 2  # Số câu tối đa trong một lượt GPU (1–4)

(APP_DIR / 'omnivoice_webui_server.py').write_text('MAX_BATCH_SIZE = ' + str(max(1, min(4, int(MAX_BATCH_SIZE)))) + '\n' + '"""One OmniVoice model shared by the browser studio and ReviewTrans REST API."""\nfrom contextlib import asynccontextmanager\nfrom concurrent.futures import Future\nfrom queue import Empty, Full, Queue\nfrom io import BytesIO\nimport json\nimport logging\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport shutil\nimport threading\nimport time\nfrom urllib.parse import urlsplit\n\nfrom fastapi import Depends, FastAPI, File, Form, HTTPException, Request, Response, UploadFile\nfrom fastapi.responses import HTMLResponse\nfrom fastapi.security import HTTPAuthorizationCredentials, HTTPBearer\nfrom pydantic import BaseModel, ConfigDict, Field, SecretStr, field_validator\nimport soundfile as sf\n\nlog = logging.getLogger("omnivoice-studio")\nVOICE_ID = re.compile(r"[a-z0-9][a-z0-9_-]{0,47}\\Z")\nRESERVED = {"auto", "female", "male", "cloned"}\nMAX_AUDIO_BYTES = 10 * 1024 * 1024\nSESSION_COOKIE = "omnivoice_ui_session"\nSESSION_SECONDS = 8 * 60 * 60\nTICKET_SECONDS = 5 * 60\n\n\nclass LoginRequest(BaseModel):\n    model_config = ConfigDict(extra="forbid")\n    api_key: SecretStr | None = None\n    ticket: SecretStr | None = None\n\n\nclass Settings(BaseModel):\n    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)\n    language: str = Field(default="vi", min_length=2, max_length=40)\n    num_step: int = Field(default=16, ge=4, le=64)\n    guidance_scale: float = Field(default=2.0, ge=0, le=5, allow_inf_nan=False)\n    denoise: bool = True\n    preprocess_prompt: bool = True\n    postprocess_output: bool = True\n\n\nclass SpeechRequest(BaseModel):\n    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)\n    model: str = "omnivoice"\n    input: str = Field(min_length=1, max_length=1000)\n    voice: str = "auto"\n    response_format: str = "wav"\n    speed: float = Field(default=1.0, ge=0.25, le=4, allow_inf_nan=False)\n    instructions: str = Field(default="", max_length=300)\n\n    @field_validator("input")\n    @classmethod\n    def nonempty(cls, value):\n        if not value.strip():\n            raise ValueError("Văn bản không được để trống.")\n        return value\n\n\nclass PreviewRequest(SpeechRequest):\n    language: str = Field(default="vi", min_length=2, max_length=40)\n    num_step: int = Field(default=16, ge=4, le=64)\n    guidance_scale: float = Field(default=2, ge=0, le=5, allow_inf_nan=False)\n    duration: float | None = Field(default=None, gt=0, le=120, allow_inf_nan=False)\n    denoise: bool = True\n    preprocess_prompt: bool = True\n    postprocess_output: bool = True\n\n\nclass VoiceMetadata(BaseModel):\n    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)\n    voice_id: str = Field(pattern=r"^[a-z0-9][a-z0-9_-]{0,47}$")\n    name: str = Field(min_length=1, max_length=80)\n    ref_text: str = Field(min_length=1, max_length=2000)\n\n\ndef write_json(path, data):\n    temporary = path.with_name(path.name + "." + secrets.token_hex(6) + ".tmp")\n    try:\n        temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef load_engine():\n    import torch\n    from omnivoice import OmniVoice\n    if not torch.cuda.is_available():\n        raise RuntimeError("Chọn GPU runtime trên Colab trước khi chạy.")\n    engine = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)\n    engine.eval()\n    return engine\n\n\nclass SpeechBatcher:\n    def __init__(self, engine, gpu_lock, limit):\n        self.engine = engine\n        self.gpu_lock = gpu_lock\n        self.limit = limit\n        self.pending = Queue(maxsize=16)\n        self.thread = threading.Thread(target=self.run, name="omnivoice-batch", daemon=True)\n        self.thread.start()\n\n    def submit(self, parameters):\n        result = Future()\n        try:\n            self.pending.put_nowait((parameters, result))\n        except Full:\n            raise HTTPException(429, "Hàng đợi giọng nói đã đầy.", headers={"Retry-After": "3"})\n        return result.result()\n\n    def close(self):\n        self.pending.put(None)\n        self.thread.join()\n\n    def run(self):\n        while True:\n            first = self.pending.get()\n            if first is None:\n                return\n            groups = {}\n            items = [first]\n            deadline = time.monotonic() + 0.04\n            closing = False\n            while len(items) < self.limit:\n                try:\n                    item = self.pending.get(timeout=max(0, deadline - time.monotonic()))\n                except Empty:\n                    break\n                if item is None:\n                    closing = True\n                    break\n                items.append(item)\n            for parameters, result in items:\n                # Chỉ gom yêu cầu có cùng thiết lập suy luận và cùng chế độ giọng.\n                key = tuple((k, parameters[k]) for k in sorted(parameters)\n                            if k not in ("text", "instruct", "voice_clone_prompt"))\n                mode = "clone" if "voice_clone_prompt" in parameters else "instruct" if "instruct" in parameters else "auto"\n                groups.setdefault((mode, key), []).append((parameters, result))\n            for group in groups.values():\n                self.generate(group)\n            if closing:\n                return\n\n    def generate(self, group):\n        try:\n            arguments = dict(group[0][0])\n            if len(group) > 1:\n                for key in ("text", "voice_clone_prompt", "instruct"):\n                    if key in arguments:\n                        arguments[key] = [params[key] for params, _ in group]\n            with self.gpu_lock:\n                audio = self.engine.generate(**arguments)\n            if len(audio) != len(group):\n                raise ValueError("Số đoạn âm thanh trả về khác số câu trong lô.")\n        except Exception as exc:\n            if len(group) == 1:\n                group[0][1].set_exception(exc)\n                return\n            log.warning("Không tạo được lô %s câu; thử lại riêng từng câu: %s", len(group), type(exc).__name__)\n        else:\n            for (_, result), waveform in zip(group, audio):\n                result.set_result(waveform)\n            return\n        for item in group:\n            self.generate([item])\n\n\ndef create_app(engine=None, api_key=None, data_dir=None, language=None, num_step=None):\n    gpu_lock = threading.Lock()\n    state_lock = threading.RLock()\n    bearer = HTTPBearer(auto_error=False)\n\n    @asynccontextmanager\n    async def lifespan(app):\n        key = api_key if api_key is not None else os.environ.get("OMNIVOICE_API_KEY", "")\n        if len(key) < 24:\n            raise RuntimeError("OMNIVOICE_API_KEY phải có ít nhất 24 ký tự.")\n        app.state.key = key\n        app.state.sessions = {}\n        app.state.login_tickets = {}\n        app.state.engine = engine if engine is not None else load_engine()\n        directory = data_dir if data_dir is not None else os.environ.get("OMNIVOICE_DATA_DIR")\n        app.state.data_dir = Path(directory) if directory else None\n        if app.state.data_dir:\n            (app.state.data_dir / "voices").mkdir(parents=True, exist_ok=True)\n        settings = Settings(language=language or os.environ.get("OMNIVOICE_LANGUAGE", "vi"),\n                            num_step=num_step or int(os.environ.get("OMNIVOICE_NUM_STEP", "16")))\n        if app.state.data_dir and (app.state.data_dir / "settings.json").is_file():\n            try:\n                settings = Settings.model_validate_json((app.state.data_dir / "settings.json").read_text(encoding="utf-8"))\n            except (ValueError, OSError):\n                log.exception("Không đọc được thiết lập cũ; dùng mặc định notebook")\n        app.state.settings = settings\n        app.state.warnings = []\n        registry = {\n            "auto": {"name": "Tự chọn giọng", "kind": "auto", "instruct": ""},\n            "female": {"name": "Nữ · voice design", "kind": "design", "instruct": "female"},\n            "male": {"name": "Nam · voice design", "kind": "design", "instruct": "male"},\n        }\n        if app.state.data_dir:\n            for metadata_path in sorted((app.state.data_dir / "voices").glob("*/voice.json")):\n                try:\n                    metadata = VoiceMetadata.model_validate_json(metadata_path.read_text(encoding="utf-8"))\n                    if metadata.voice_id in RESERVED or metadata.voice_id != metadata_path.parent.name:\n                        raise ValueError("Mã giọng không hợp lệ")\n                    prompt = app.state.engine.create_voice_clone_prompt(\n                        ref_audio=str(metadata_path.parent / "reference.wav"), ref_text=metadata.ref_text)\n                    registry[metadata.voice_id] = {"name": metadata.name, "kind": "clone", "prompt": prompt}\n                except Exception:\n                    log.exception("Không nạp được giọng %s", metadata_path.parent.name)\n                    app.state.warnings.append("Không nạp được giọng " + metadata_path.parent.name + "; xem server.log.")\n        reference = os.environ.get("OMNIVOICE_REFERENCE", "")\n        transcript = os.environ.get("OMNIVOICE_REFERENCE_TEXT", "").strip()\n        if reference and transcript:\n            prompt = app.state.engine.create_voice_clone_prompt(ref_audio=reference, ref_text=transcript)\n            registry = {"cloned": {"name": "Mẫu từ ô notebook (tạm thời)", "kind": "clone", "prompt": prompt}, **registry}\n        app.state.voices = registry\n        app.state.batcher = SpeechBatcher(app.state.engine, gpu_lock, MAX_BATCH_SIZE)\n        try:\n            yield\n        finally:\n            app.state.batcher.close()\n\n    app = FastAPI(title="OmniVoice Studio · ReviewTrans", version="2.1.0", lifespan=lifespan)\n    app.state.generation_lock = gpu_lock\n\n    def key_matches(value):\n        return secrets.compare_digest(value.encode("utf-8"), app.state.key.encode("utf-8"))\n\n    def authorize(credentials: HTTPAuthorizationCredentials | None = Depends(bearer)):\n        if credentials is None or not key_matches(credentials.credentials):\n            raise HTTPException(401, "API key không đúng.", headers={"WWW-Authenticate": "Bearer"})\n\n    auth = [Depends(authorize)]\n\n    def same_origin(request):\n        origin = request.headers.get("origin")\n        if origin:\n            parsed = urlsplit(origin)\n            if parsed.scheme not in {"http", "https"} or parsed.netloc != request.headers.get("host"):\n                raise HTTPException(403, "Hãy mở Web UI từ đúng URL của phiên Colab.")\n\n    def prune_login_state():\n        now = time.monotonic()\n        for mapping in (app.state.sessions, app.state.login_tickets):\n            for value, expiry in list(mapping.items()):\n                if expiry <= now:\n                    mapping.pop(value, None)\n            while len(mapping) >= 128:\n                mapping.pop(next(iter(mapping)))\n\n    def authorize_ui(request: Request, credentials: HTTPAuthorizationCredentials | None = Depends(bearer)):\n        # Bearer remains available for notebook checks; the browser uses an HttpOnly session cookie.\n        if credentials is not None and key_matches(credentials.credentials):\n            return\n        session_id = request.cookies.get(SESSION_COOKIE, "")\n        with state_lock:\n            valid = app.state.sessions.get(session_id, 0) > time.monotonic()\n        if not valid:\n            raise HTTPException(401, "Phiên Web UI chưa đăng nhập hoặc đã hết hạn.")\n        if request.method not in {"GET", "HEAD", "OPTIONS"}:\n            if not request.headers.get("origin"):\n                raise HTTPException(403, "Thiếu Origin của Web UI.")\n            same_origin(request)\n\n    ui_auth = [Depends(authorize_ui)]\n\n    @app.post("/ui/launch", dependencies=auth)\n    def launch_ui():\n        with state_lock:\n            prune_login_state()\n            ticket = secrets.token_urlsafe(32)\n            app.state.login_tickets[ticket] = time.monotonic() + TICKET_SECONDS\n        return Response(json.dumps({"ticket": ticket, "expires_in": TICKET_SECONDS}),\n                        media_type="application/json", headers={"Cache-Control": "no-store"})\n\n    @app.post("/ui/login")\n    def login_ui(body: LoginRequest, request: Request, response: Response):\n        same_origin(request)\n        if (body.api_key is None) == (body.ticket is None):\n            raise HTTPException(400, "Cần API key hoặc mã đăng nhập dùng một lần.")\n        if body.ticket is not None:\n            with state_lock:\n                expiry = app.state.login_tickets.pop(body.ticket.get_secret_value(), 0)\n            if expiry <= time.monotonic():\n                raise HTTPException(401, "Liên kết đăng nhập đã dùng hoặc hết hạn. Lấy liên kết mới trong Colab.")\n        elif not key_matches(body.api_key.get_secret_value().strip()):\n            raise HTTPException(401, "API key không đúng. Có thể mở bằng liên kết đăng nhập trực tiếp từ Colab.")\n        with state_lock:\n            prune_login_state()\n            old_session = request.cookies.get(SESSION_COOKIE, "")\n            app.state.sessions.pop(old_session, None)\n            session_id = secrets.token_urlsafe(32)\n            app.state.sessions[session_id] = time.monotonic() + SESSION_SECONDS\n        response.set_cookie(SESSION_COOKIE, session_id, max_age=SESSION_SECONDS, path="/ui",\n                            httponly=True, secure=request.url.scheme == "https", samesite="strict")\n        response.headers["Cache-Control"] = "no-store"\n        return {"authenticated": True, "expires_in": SESSION_SECONDS}\n\n    @app.post("/ui/logout")\n    def logout_ui(request: Request):\n        same_origin(request)\n        with state_lock:\n            app.state.sessions.pop(request.cookies.get(SESSION_COOKIE, ""), None)\n        response = Response(status_code=204, headers={"Cache-Control": "no-store"})\n        response.delete_cookie(SESSION_COOKIE, path="/ui", httponly=True, samesite="strict")\n        return response\n\n    def languages():\n        method = getattr(app.state.engine, "supported_language_ids", None)\n        return sorted(method()) if method else ["de", "en", "es", "fr", "ja", "ko", "vi", "zh"]\n\n    def check_language(value):\n        if value not in languages():\n            raise HTTPException(400, "Ngôn ngữ không được hỗ trợ; chọn từ danh sách.")\n\n    def voice_items():\n        with state_lock:\n            return [{"id": ident, "name": item["name"], "kind": item["kind"],\n                     "language": app.state.settings.language} for ident, item in app.state.voices.items()]\n\n    @app.get("/", response_class=HTMLResponse, include_in_schema=False)\n    def studio():\n        return HTMLResponse(Path(__file__).with_name("omnivoice_webui.html").read_text(encoding="utf-8"),\n                            headers={"Cache-Control": "no-store", "X-Content-Type-Options": "nosniff",\n                                     "Referrer-Policy": "no-referrer"})\n\n    @app.get("/health", dependencies=auth)\n    def health():\n        with state_lock:\n            return {"status": "ready", "model": "omnivoice", **app.state.settings.model_dump(),\n                    "sample_rate": app.state.engine.sampling_rate, "voice_count": len(app.state.voices),\n                    "ui_auth": "session-v1"}\n\n    @app.get("/v1/models", dependencies=auth)\n    def models():\n        return {"object": "list", "data": [{"id": "omnivoice", "object": "model", "owned_by": "k2-fsa"}]}\n\n    @app.get("/v1/voices", dependencies=auth)\n    def voices():\n        return {"voices": voice_items()}\n\n    @app.get("/ui/state", dependencies=ui_auth)\n    def state():\n        with state_lock:\n            return {"settings": app.state.settings.model_dump(), "voices": voice_items(),\n                    "languages": languages(), "storage": str(app.state.data_dir or "Chỉ trong phiên"),\n                    "warnings": list(app.state.warnings)}\n\n    @app.put("/ui/settings", dependencies=ui_auth)\n    def save_settings(body: Settings):\n        check_language(body.language)\n        with state_lock:\n            if app.state.data_dir:\n                try:\n                    write_json(app.state.data_dir / "settings.json", body.model_dump())\n                except OSError as exc:\n                    raise HTTPException(507, "Không lưu được thiết lập. Kiểm tra kết nối/dung lượng Drive.") from exc\n            app.state.settings = body\n        return {"settings": body.model_dump(), "persisted": app.state.data_dir is not None}\n\n    @app.post("/ui/voices", dependencies=ui_auth)\n    def add_voice(voice_id: str = Form(..., min_length=1, max_length=48),\n                  name: str = Form(..., min_length=1, max_length=80),\n                  ref_text: str = Form(..., min_length=1, max_length=2000),\n                  audio: UploadFile = File(...)):\n        voice_id, name, ref_text = voice_id.strip(), name.strip(), ref_text.strip()\n        if not VOICE_ID.fullmatch(voice_id) or voice_id in RESERVED:\n            raise HTTPException(400, "Mã giọng: chữ thường a-z, số, _ hoặc -; không dùng auto/female/male/cloned.")\n        if not name or not ref_text:\n            raise HTTPException(400, "Cần tên giọng và lời thoại chính xác của mẫu.")\n        root = app.state.data_dir\n        if root is None:\n            raise HTTPException(503, "Chưa cấu hình thư mục lưu giọng. Chạy lại notebook từ phần 1.")\n        if not gpu_lock.acquire(blocking=False):\n            raise HTTPException(429, "GPU đang bận. Chờ tạo giọng xong rồi thêm mẫu.")\n        staging = None\n        try:\n            with state_lock:\n                if voice_id in app.state.voices or (root / "voices" / voice_id).exists():\n                    raise HTTPException(409, "Mã giọng đã tồn tại. Chọn mã khác.")\n            content = audio.file.read(MAX_AUDIO_BYTES + 1)\n            if len(content) > MAX_AUDIO_BYTES:\n                raise HTTPException(413, "File mẫu không được quá 10 MB.")\n            try:\n                info = sf.info(BytesIO(content))\n            except (RuntimeError, ValueError) as exc:\n                raise HTTPException(400, "Không đọc được âm thanh. Chọn file WAV hoặc FLAC.") from exc\n            if info.format not in {"WAV", "WAVEX", "FLAC"}:\n                raise HTTPException(400, "Chỉ nhận WAV hoặc FLAC.")\n            if not 1 <= info.duration <= 15:\n                raise HTTPException(400, "Mẫu cần dài 1–15 giây; nên dùng 3–10 giây.")\n            if not 1 <= info.channels <= 2 or not 8000 <= info.samplerate <= 96000:\n                raise HTTPException(400, "Mẫu cần mono/stereo, tần số 8–96 kHz.")\n            waveform, sample_rate = sf.read(BytesIO(content), dtype="float32", always_2d=True)\n            staging = root / "voices" / (".upload-" + secrets.token_hex(10))\n            staging.mkdir()\n            audio_path = staging / "reference.wav"\n            sf.write(str(audio_path), waveform.mean(axis=1), sample_rate, subtype="PCM_16")\n            prompt = app.state.engine.create_voice_clone_prompt(ref_audio=str(audio_path), ref_text=ref_text)\n            metadata = VoiceMetadata(voice_id=voice_id, name=name, ref_text=ref_text)\n            write_json(staging / "voice.json", metadata.model_dump())\n            destination = root / "voices" / voice_id\n            os.rename(staging, destination)\n            staging = None\n            with state_lock:\n                app.state.voices[voice_id] = {"name": name, "kind": "clone", "prompt": prompt}\n            return {"voice": {"id": voice_id, "name": name, "kind": "clone"},\n                    "saved": True, "message": "Đã lưu và thêm vào API. ReviewTrans có thể cần làm mới cache giọng."}\n        except OSError as exc:\n            log.exception("Không lưu được giọng")\n            raise HTTPException(507, "Không lưu được giọng. Kiểm tra Drive và dung lượng trống.") from exc\n        except (RuntimeError, ValueError) as exc:\n            log.exception("Không tạo được prompt clone")\n            raise HTTPException(500, "Không tạo được giọng tham chiếu; xem log Colab.") from exc\n        finally:\n            audio.file.close()\n            if staging is not None:\n                try:\n                    if staging.name.startswith(".upload-") and staging.resolve().parent == (root / "voices").resolve():\n                        shutil.rmtree(staging)\n                except OSError:\n                    log.warning("Còn thư mục upload dở: %s", staging.name)\n            gpu_lock.release()\n\n    def synthesize(body, preview=False):\n        if body.model not in {"omnivoice", "k2-fsa/OmniVoice"}:\n            raise HTTPException(400, "Chọn model omnivoice.")\n        if body.response_format != "wav":\n            raise HTTPException(400, "Chọn định dạng wav trong ReviewTrans.")\n        try:\n            with state_lock:\n                item = app.state.voices.get(body.voice)\n                parameters = app.state.settings.model_dump()\n            if item is None:\n                raise HTTPException(400, "Giọng không tồn tại. Lấy lại danh sách /v1/voices.")\n            if "prompt" in item and body.instructions:\n                raise HTTPException(400, "Để trống chỉ dẫn giọng đọc khi dùng voice clone.")\n            if preview:\n                for field in Settings.model_fields:\n                    parameters[field] = getattr(body, field)\n                if body.duration is not None:\n                    parameters["duration"] = body.duration\n            check_language(parameters["language"])\n            parameters.update(text=body.input, speed=body.speed)\n            if "prompt" in item:\n                parameters["voice_clone_prompt"] = item["prompt"]\n            elif body.instructions or item.get("instruct"):\n                parameters["instruct"] = body.instructions or item["instruct"]\n            waveform = app.state.batcher.submit(parameters)\n            output = BytesIO()\n            sf.write(output, waveform, int(app.state.engine.sampling_rate), format="WAV", subtype="PCM_16")\n            return Response(output.getvalue(), media_type="audio/wav", headers={\n                "Content-Disposition": \'attachment; filename="speech.wav"\', "Cache-Control": "no-store"})\n        except ValueError as exc:\n            log.exception("Tham số sinh giọng không hợp lệ")\n            raise HTTPException(400, "OmniVoice từ chối tham số/chỉ dẫn. Kiểm tra giá trị và server.log.") from exc\n        except RuntimeError as exc:\n            log.exception("Suy luận thất bại")\n            if "out of memory" in str(exc).lower():\n                import torch\n                torch.cuda.empty_cache()\n                raise HTTPException(503, "GPU hết VRAM. Chia câu ngắn hơn, số luồng 1.") from exc\n            raise HTTPException(500, "Không tạo được âm thanh; xem log Colab.") from exc\n\n    @app.post("/v1/audio/speech", dependencies=auth, response_class=Response,\n              responses={200: {"content": {"audio/wav": {}}, "description": "PCM16 WAV"}})\n    def speech(body: SpeechRequest):\n        return synthesize(body)\n\n    @app.post("/ui/preview", dependencies=ui_auth, response_class=Response)\n    def preview(body: PreviewRequest):\n        return synthesize(body, preview=True)\n\n    return app\n\n\napp = create_app()\n', encoding='utf-8')
(APP_DIR / 'omnivoice_webui.html').write_text('<!doctype html>\n<html lang="vi">\n<head>\n<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>OmniVoice Studio</title>\n<style>\n:root{color-scheme:dark;--bg:#0c111b;--panel:#141c29;--line:#293548;--ink:#eef3fc;--muted:#a7b5ca;--brand:#7ce5cf;--accent:#b9a2ff;--error:#ffb4b4}\n*{box-sizing:border-box}body{margin:0;font:15px/1.55 "Segoe UI",system-ui,sans-serif;background:var(--bg);color:var(--ink)}\nbutton,input,select,textarea{font:inherit}button{cursor:pointer;border:1px solid var(--line);border-radius:9px;padding:10px 16px;background:#202d40;color:var(--ink);font-weight:600}\nbutton.primary{background:var(--brand);color:#0c2b25;border-color:var(--brand)}button:hover{filter:brightness(1.09)}button:disabled{opacity:.5;cursor:wait}button:focus-visible,a:focus-visible{outline:3px solid var(--accent);outline-offset:3px}\ninput,select,textarea{width:100%;background:#0f1622;color:var(--ink);border:1px solid #3a4961;border-radius:8px;padding:10px 12px;min-width:0}input:focus,textarea:focus,select:focus{outline:2px solid var(--brand);outline-offset:1px}input[type=range]{padding:0;accent-color:var(--brand)}input[type=checkbox]{width:auto;accent-color:var(--brand)}textarea{resize:vertical}label{display:block;font-size:13px;color:#d7e1f1;font-weight:600;margin:0 0 7px}\n.wrap{max-width:1220px;margin:auto;padding:30px 24px 48px}header{display:flex;justify-content:space-between;align-items:center;gap:20px;margin-bottom:26px}h1{font-size:28px;letter-spacing:-.8px;line-height:1.2;margin:0 0 6px}h2{font-size:18px;letter-spacing:-.3px;margin:0 0 6px}p{margin:0 0 16px}.muted,.hint{color:var(--muted)}.hint{font-size:12px;margin:7px 0 0}.eyebrow{font-size:11px;letter-spacing:2px;color:var(--brand);font-weight:700;margin-bottom:10px}\n.badge{display:inline-flex;align-items:center;gap:8px;font-size:12px;color:var(--muted);border:1px solid var(--line);border-radius:30px;padding:7px 12px;white-space:nowrap}.dot{width:7px;height:7px;background:#6c7b92;border-radius:50%}.connected .dot{background:var(--brand)}.connected{color:var(--brand)}\n.card{background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:22px}.login{max-width:620px;margin:70px auto}.row{display:flex;gap:12px;align-items:center}.row>*{min-width:0}.grow{flex:1}.grid{display:grid;grid-template-columns:minmax(0,1.6fr) minmax(280px,1fr);gap:20px}.stack{display:flex;flex-direction:column;gap:20px}.field{margin-top:18px}.fields{display:grid;grid-template-columns:1fr 1fr;gap:16px}.section-head{display:flex;align-items:center;justify-content:space-between;gap:12px;margin-bottom:16px}.section-num{font-size:12px;color:var(--brand);margin-right:9px}small{font-size:12px;color:var(--muted)}.checks{display:grid;gap:10px;margin-top:18px}.check{display:flex;align-items:center;gap:8px;font-weight:400;font-size:13px;margin:0}.stat{display:flex;justify-content:space-between;margin-bottom:8px}output{font-size:13px;color:var(--brand)}.notice{padding:12px 15px;border:1px solid #365649;background:#10261f;border-radius:9px;color:#c1f3df;font-size:13px;white-space:pre-wrap;overflow-wrap:anywhere;margin-bottom:20px}.notice.error{border-color:#6b3c49;background:#301b25;color:var(--error)}.notice:empty{display:none}.subtle{font-size:12px;background:#0f1622;padding:12px;border-radius:8px;color:var(--muted)}.actions{display:flex;gap:10px;align-items:center;margin-top:20px}.text-button{background:none;border:none;padding:6px;color:var(--muted);font-size:12px}.result{border-top:1px solid var(--line);padding-top:18px;margin-top:20px}audio{width:100%;margin:8px 0}.download{color:var(--brand);font-size:13px}.preview-ref{margin-top:10px}.footer{display:flex;justify-content:space-between;gap:16px;margin-top:24px;font-size:12px;color:var(--muted)}[hidden]{display:none!important}@media(max-width:800px){.wrap{padding:20px 14px}.grid{grid-template-columns:1fr}header{align-items:flex-start}h1{font-size:24px}.card{padding:18px}.login{margin:25px auto}.footer{flex-direction:column}.fields{gap:12px}}@media(max-width:460px){.fields{grid-template-columns:1fr}.row{flex-wrap:wrap}.badge{font-size:11px}.actions{flex-wrap:wrap}}\n</style>\n</head>\n<body><main class="wrap">\n<header><div><div class="eyebrow">VOICE WORKSPACE</div><h1>OmniVoice Studio</h1><span class="muted">Tạo giọng, lưu mẫu và kết nối ReviewTrans.</span></div><div><span id="badge" class="badge"><i class="dot"></i><span id="connectionLabel">Chưa kết nối</span></span><button id="disconnect" class="text-button" hidden>Ngắt kết nối</button></div></header>\n<div id="status" class="notice" role="status" aria-live="polite"></div>\n<section id="loginPanel" class="card login">\n <h2>Kết nối phiên Colab</h2><p class="muted">Nhập API key bạn đã đặt trong notebook để quản lý giọng và tạo âm thanh.</p>\n <form id="loginForm"><label for="apiKey">API key</label><div class="row"><input class="grow" id="apiKey" type="password" autocomplete="off" spellcheck="false" required placeholder="Nhập khóa của phiên Colab"><button id="connect" class="primary" type="submit">Kết nối</button></div></form>\n <p class="hint">Có thể dùng liên kết đăng nhập trực tiếp từ Colab. Trình duyệt giữ phiên riêng; API key không được lưu trên trang.</p>\n</section>\n<div id="workspace" hidden>\n<div class="grid">\n <div class="stack">\n  <section class="card">\n   <div class="section-head"><div><h2><span class="section-num">01</span>Tạo giọng & nghe thử</h2><small>Chọn một giọng đã lưu hoặc thử voice design.</small></div><button id="refresh" class="text-button">Làm mới</button></div>\n   <form id="previewForm">\n    <div class="fields"><div><label for="voice">Giọng đọc</label><select id="voice" required></select></div><div><label for="language">Ngôn ngữ</label><select id="language" required></select></div></div>\n    <div class="field"><label for="text">Văn bản cần đọc</label><textarea id="text" rows="5" maxlength="1000" required>Xin chào, đây là giọng đọc thử từ OmniVoice.</textarea><p class="hint"><span id="charCount">0</span>/1.000 ký tự</p></div>\n    <div class="field"><label for="instructions">Mô tả giọng · tùy chọn</label><input id="instructions" maxlength="300" placeholder="Ví dụ: female, low pitch"><p id="instructionHint" class="hint">Dùng thuộc tính tiếng Anh. Voice design có thể thay đổi giữa các lần tạo.</p></div>\n    <div class="fields field"><div><label class="stat" for="speed"><span>Tốc độ</span><output id="speedValue">1.00×</output></label><input id="speed" type="range" min="0.25" max="4" step="0.05" value="1"></div><div><label for="duration">Thời lượng cố định · giây</label><input id="duration" type="number" min="0.1" max="120" step="0.1" placeholder="Tự ước lượng"><p class="hint">Chỉ áp dụng khi nghe thử; ưu tiên hơn tốc độ.</p></div></div>\n    <div class="actions"><button id="generate" class="primary" type="submit">Tạo âm thanh</button><small id="generateStatus"></small></div>\n   </form>\n   <div id="result" class="result" hidden><label for="outputAudio">Kết quả</label><audio id="outputAudio" controls></audio><a id="download" class="download" download="omnivoice.wav">Tải file WAV</a></div>\n  </section>\n  <section class="card">\n   <div class="section-head"><div><h2><span class="section-num">02</span>Thêm giọng tham chiếu</h2><small>Mẫu 3–10 giây, một người nói, âm thanh sạch.</small></div></div>\n   <form id="voiceForm">\n    <div class="fields"><div><label for="voiceId">Mã giọng trong ReviewTrans</label><input id="voiceId" pattern="[a-z0-9][a-z0-9_-]{0,47}" maxlength="48" required placeholder="Ví dụ: giong_nu_01"><p class="hint">Chữ thường không dấu, số, _ hoặc -. Mỗi giọng một mã riêng.</p></div><div><label for="voiceName">Tên hiển thị</label><input id="voiceName" maxlength="80" required placeholder="Ví dụ: Giọng nữ kể chuyện"></div></div>\n    <div class="field"><label for="referenceFile">File giọng mẫu · WAV / FLAC</label><input id="referenceFile" type="file" accept=".wav,.flac,audio/wav,audio/flac" required><p class="hint">Tối đa 10 MB, dài 1–15 giây. Chỉ dùng giọng bạn có quyền sử dụng.</p><audio id="referenceAudio" class="preview-ref" controls hidden></audio></div>\n    <div class="field"><label for="referenceText">Lời nói chính xác trong mẫu</label><textarea id="referenceText" rows="3" maxlength="2000" required placeholder="Nhập nguyên văn lời nói trong file vừa chọn"></textarea></div>\n    <div class="actions"><button id="addVoice" type="submit">Lưu & thêm giọng clone</button><small id="uploadStatus"></small></div>\n   </form>\n  </section>\n </div>\n <aside class="stack">\n  <section class="card">\n   <h2>Thông số tạo giọng</h2><p class="hint">Nghe thử dùng các giá trị đang hiển thị. Bấm lưu để áp dụng cho yêu cầu mới từ ReviewTrans.</p>\n   <div class="field"><label class="stat" for="steps"><span>Số bước sinh</span><output id="stepsValue">16</output></label><input id="steps" type="range" min="4" max="64" step="1" value="16"><p class="hint">16 để thử nhanh; 32 khi muốn thử chất lượng cao hơn.</p></div>\n   <div class="field"><label class="stat" for="guidance"><span>Guidance scale</span><output id="guidanceValue">2.0</output></label><input id="guidance" type="range" min="0" max="5" step="0.1" value="2"></div>\n   <div class="checks"><label class="check"><input id="denoise" type="checkbox" checked>Làm sạch âm thanh</label><label class="check"><input id="preprocess" type="checkbox" checked>Xử lý giọng mẫu</label><label class="check"><input id="postprocess" type="checkbox" checked>Xử lý âm thanh đầu ra</label></div>\n   <div class="actions"><button id="saveSettings">Lưu thiết lập cho API</button></div><p class="hint">Lưu ngôn ngữ đang chọn và các thông số trong khung này. Tốc độ và giọng đọc vẫn do ReviewTrans chọn cho từng yêu cầu.</p>\n  </section>\n  <section class="card"><h2>Kết nối ReviewTrans</h2><div class="field"><label for="baseUrl">Base URL</label><input id="baseUrl" readonly spellcheck="false"><button id="copyUrl" class="text-button">Sao chép URL</button></div><div class="subtle">Loại: API kiểu OpenAI<br>Model: <strong>omnivoice</strong><br>Định dạng: <strong>wav</strong> · Số luồng: <strong>__BATCH_SIZE__</strong><br>API key: dùng khóa bạn vừa nhập.</div><p class="hint">Giọng mới được thêm vào API ngay. Nếu ReviewTrans chưa hiển thị, khởi động lại ứng dụng để làm mới cache giọng.</p></section>\n  <section class="card"><h2>Lưu trữ</h2><p id="storage" class="hint"></p><p class="hint">Giọng mẫu và thiết lập được lưu tại đây. Model và giao diện dùng chung một GPU; khi đang tạo giọng, yêu cầu khác cần chờ.</p></section>\n </aside>\n</div><footer class="footer"><span>OmniVoice · phiên Colab phải đang hoạt động</span><span>Âm thanh WAV · Web UI và ReviewTrans dùng chung model</span></footer>\n</div></main>\n<script>\n\'use strict\';\nconst $=id=>document.getElementById(id);\nlet voices=[], outputUrl=\'\', referenceUrl=\'\', busy=false;\nfunction message(text,error=false){$(\'status\').textContent=text;$(\'status\').classList.toggle(\'error\',error)}\nfunction setBusy(value){busy=value;for(const id of [\'generate\',\'addVoice\'])$(id).disabled=value}\nasync function logout(){try{await request(\'/ui/logout\',{method:\'POST\'})}catch(error){message(error.message,true);return}$(\'apiKey\').value=\'\';$(\'workspace\').hidden=true;$(\'loginPanel\').hidden=false;$(\'disconnect\').hidden=true;$(\'badge\').classList.remove(\'connected\');$(\'connectionLabel\').textContent=\'Chưa kết nối\';if(outputUrl){URL.revokeObjectURL(outputUrl);outputUrl=\'\'}$(\'outputAudio\').removeAttribute(\'src\');$(\'result\').hidden=true;message(\'Đã ngắt kết nối và kết thúc phiên Web UI.\')}\nasync function request(path,{method=\'GET\',body=null}={}){\n const headers={\'ngrok-skip-browser-warning\':\'1\'};\n if(body!==null && !(body instanceof FormData)){headers[\'Content-Type\']=\'application/json\';body=JSON.stringify(body)}\n let response;\n try{response=await fetch(path,{method,headers,body,cache:\'no-store\',credentials:\'same-origin\'})}catch{throw new Error(\'Không kết nối được Colab. Kiểm tra runtime và đường dẫn ngrok.\')}\n if(!response.ok){let detail=\'\';try{const data=await response.json();detail=typeof data.detail===\'string\'?data.detail:JSON.stringify(data.detail)}catch{detail=await response.text().catch(()=> \'\')}const error=new Error(\'HTTP \'+response.status+\': \'+(detail||\'Không thực hiện được yêu cầu.\').slice(0,600));error.status=response.status;throw error}\n return response;\n}\nfunction values(){return {language:$(\'language\').value,num_step:Number($(\'steps\').value),guidance_scale:Number($(\'guidance\').value),denoise:$(\'denoise\').checked,preprocess_prompt:$(\'preprocess\').checked,postprocess_output:$(\'postprocess\').checked}}\nfunction voiceMode(){const clone=voices.find(v=>v.id===$(\'voice\').value)?.kind===\'clone\';$(\'instructions\').disabled=clone;if(clone)$(\'instructions\').value=\'\';$(\'instructionHint\').textContent=clone?\'Đang dùng giọng tham chiếu đã lưu. Không cần nhập mô tả giọng.\':\'Dùng thuộc tính tiếng Anh. Voice design có thể thay đổi giữa các lần tạo.\'}\nfunction updateValues(){$(\'speedValue\').textContent=Number($(\'speed\').value).toFixed(2)+\'×\';$(\'stepsValue\').textContent=$(\'steps\').value;$(\'guidanceValue\').textContent=Number($(\'guidance\').value).toFixed(1);$(\'charCount\').textContent=$(\'text\').value.length}\nasync function refresh(preferred=\'\'){\n const data=await(await request(\'/ui/state\')).json();\n const selected=preferred||$(\'voice\').value;voices=data.voices;$(\'voice\').replaceChildren();\n for(const item of voices){const option=document.createElement(\'option\');option.value=item.id;option.textContent=item.name+\' · \'+item.id;$(\'voice\').append(option)}\n $(\'voice\').value=voices.some(v=>v.id===selected)?selected:(voices.find(v=>v.kind===\'clone\')?.id||\'auto\');\n $(\'language\').replaceChildren();\n let names;try{names=new Intl.DisplayNames([\'vi\'],{type:\'language\'})}catch{}\n for(const code of data.languages){const option=document.createElement(\'option\');option.value=code;let label=code;try{label=names?.of(code)||code}catch{}option.textContent=label+\' (\'+code+\')\';$(\'language\').append(option)}\n const s=data.settings;$(\'language\').value=s.language;$(\'steps\').value=s.num_step;$(\'guidance\').value=s.guidance_scale;$(\'denoise\').checked=s.denoise;$(\'preprocess\').checked=s.preprocess_prompt;$(\'postprocess\').checked=s.postprocess_output;\n $(\'storage\').textContent=data.storage;$(\'baseUrl\').value=location.origin+\'/v1\';updateValues();voiceMode();\n if(data.warnings.length)message(data.warnings.join(\'\\n\'),true);\n return data;\n}\nfunction showWorkspace(state){$(\'apiKey\').value=\'\';$(\'workspace\').hidden=false;$(\'loginPanel\').hidden=true;$(\'disconnect\').hidden=false;$(\'badge\').classList.add(\'connected\');$(\'connectionLabel\').textContent=\'Đã kết nối\';if(!state.warnings.length)message(\'Đã kết nối. Bạn có thể tạo âm thanh hoặc thêm giọng mẫu.\')}\nasync function login(body){await request(\'/ui/login\',{method:\'POST\',body});const state=await refresh();showWorkspace(state)}\n$(\'loginForm\').addEventListener(\'submit\',async event=>{event.preventDefault();$(\'connect\').disabled=true;try{await login({api_key:$(\'apiKey\').value.trim()})}catch(error){message(error.message,true)}finally{$(\'connect\').disabled=false}});\n$(\'disconnect\').addEventListener(\'click\',logout);\n$(\'refresh\').addEventListener(\'click\',async()=>{try{await refresh();message(\'Đã làm mới danh sách giọng và thiết lập đã lưu.\')}catch(error){message(error.message,true)}});\n$(\'voice\').addEventListener(\'change\',voiceMode);\nfor(const id of [\'speed\',\'steps\',\'guidance\',\'text\'])$(id).addEventListener(\'input\',updateValues);\n$(\'saveSettings\').addEventListener(\'click\',async()=>{$(\'saveSettings\').disabled=true;try{const data=await(await request(\'/ui/settings\',{method:\'PUT\',body:values()})).json();message(data.persisted?\'Đã lưu thiết lập. Yêu cầu mới từ ReviewTrans sẽ dùng các giá trị này.\':\'Đã áp dụng trong phiên hiện tại; chưa cấu hình lưu lâu dài.\')}catch(error){message(error.message,true)}finally{$(\'saveSettings\').disabled=false}});\n$(\'previewForm\').addEventListener(\'submit\',async event=>{event.preventDefault();if(busy)return;setBusy(true);$(\'generateStatus\').textContent=\'Đang tạo âm thanh…\';message(\'\');const started=performance.now();try{const duration=$(\'duration\').value;const payload={...values(),model:\'omnivoice\',input:$(\'text\').value,voice:$(\'voice\').value,response_format:\'wav\',speed:Number($(\'speed\').value),instructions:$(\'instructions\').disabled?\'\':$(\'instructions\').value,duration:duration===\'\'?null:Number(duration)};const response=await request(\'/ui/preview\',{method:\'POST\',body:payload});if(!(response.headers.get(\'Content-Type\')||\'\').includes(\'audio/\'))throw new Error(\'Máy chủ không trả về âm thanh.\');const blob=await response.blob();if(outputUrl)URL.revokeObjectURL(outputUrl);outputUrl=URL.createObjectURL(blob);$(\'outputAudio\').src=outputUrl;$(\'download\').href=outputUrl;$(\'result\').hidden=false;$(\'generateStatus\').textContent=\'Hoàn tất · \'+((performance.now()-started)/1000).toFixed(1)+\' giây\'}catch(error){$(\'generateStatus\').textContent=\'Chưa tạo được âm thanh\';message(error.message,true)}finally{setBusy(false)}});\n$(\'referenceFile\').addEventListener(\'change\',()=>{if(referenceUrl)URL.revokeObjectURL(referenceUrl);const file=$(\'referenceFile\').files[0];$(\'referenceAudio\').hidden=!file;if(file){referenceUrl=URL.createObjectURL(file);$(\'referenceAudio\').src=referenceUrl}});\n$(\'voiceForm\').addEventListener(\'submit\',async event=>{event.preventDefault();if(busy)return;const file=$(\'referenceFile\').files[0];if(!file)return;if(file.size>10*1024*1024){message(\'File mẫu không được quá 10 MB.\',true);return}const body=new FormData();body.append(\'voice_id\',$(\'voiceId\').value.trim());body.append(\'name\',$(\'voiceName\').value.trim());body.append(\'ref_text\',$(\'referenceText\').value.trim());body.append(\'audio\',file);setBusy(true);$(\'uploadStatus\').textContent=\'Đang lưu và tạo giọng clone…\';try{const data=await(await request(\'/ui/voices\',{method:\'POST\',body})).json();await refresh(data.voice.id);$(\'voiceForm\').reset();$(\'referenceAudio\').hidden=true;$(\'referenceAudio\').removeAttribute(\'src\');if(referenceUrl){URL.revokeObjectURL(referenceUrl);referenceUrl=\'\'}$(\'uploadStatus\').textContent=\'Đã lưu: \'+data.voice.id;message(data.message+\' Mã giọng: \'+data.voice.id)}catch(error){$(\'uploadStatus\').textContent=\'Chưa lưu được giọng\';message(error.message,true)}finally{setBusy(false)}});\n$(\'copyUrl\').addEventListener(\'click\',async()=>{try{await navigator.clipboard.writeText($(\'baseUrl\').value);message(\'Đã sao chép Base URL.\')}catch{$(\'baseUrl\').select();message(\'Chọn và sao chép Base URL trong ô phía trên.\')}});\nwindow.addEventListener(\'pagehide\',()=>{if(outputUrl)URL.revokeObjectURL(outputUrl);if(referenceUrl)URL.revokeObjectURL(referenceUrl)});\nasync function restoreLogin(){const ticket=new URLSearchParams(location.hash.slice(1)).get(\'login\');if(ticket){history.replaceState(null,\'\',location.pathname+location.search);$(\'connect\').disabled=true;try{await login({ticket})}catch(error){message(error.message,true)}finally{$(\'connect\').disabled=false}return}try{showWorkspace(await refresh())}catch(error){if(error.status!==401)message(error.message,true)}}\nupdateValues();\nrestoreLogin();\n</script></body></html>\n'.replace('__BATCH_SIZE__', str(max(1, min(4, int(MAX_BATCH_SIZE))))), encoding='utf-8')


In [ ]:
# 5b · Khởi động server, nạp model và giọng đã lưu

# Dừng server do chính notebook này tạo nếu đang chạy, để không nạp trùng model.
if "server_process" in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait()

server_env = os.environ.copy()
server_env.update({
    "OMNIVOICE_API_KEY": API_KEY, "OMNIVOICE_LANGUAGE": LANGUAGE,
    "OMNIVOICE_NUM_STEP": str(NUM_STEP), "OMNIVOICE_REFERENCE": REFERENCE_PATH,
    "OMNIVOICE_REFERENCE_TEXT": REFERENCE_TEXT, "OMNIVOICE_DATA_DIR": str(STUDIO_DIR),
    "HF_HOME": str(APP_DIR / "hf-home"), "HF_HUB_CACHE": str(MODEL_CACHE),
    "HF_XET_CACHE": str(APP_DIR / "xet-cache"), "HF_TOKEN_PATH": str(APP_DIR / "hf-token"),
    "HF_HUB_DISABLE_SYMLINKS": "1", "TOKENIZERS_PARALLELISM": "false",
})
server_log = APP_DIR / "server.log"
with server_log.open("w") as log_file:
    server_process = subprocess.Popen([
        PY, "-u", "-m", "uvicorn", "omnivoice_webui_server:app", "--host", "127.0.0.1",
        "--port", "8000", "--workers", "1", "--no-access-log",
    ], cwd=str(APP_DIR), env=server_env, stdout=log_file, stderr=subprocess.STDOUT)

def api_call(url, payload=None, timeout=180):
    headers = {"Authorization": "Bearer " + API_KEY, "ngrok-skip-browser-warning": "1"}
    data = None
    if payload is not None:
        headers["Content-Type"] = "application/json"
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers=headers)
    return urllib.request.urlopen(req, timeout=timeout)

for attempt in range(600):
    if server_process.poll() is not None:
        print(server_log.read_text(errors="replace")[-10000:])
        raise RuntimeError("Server dừng do lỗi; đọc log ở trên.")
    try:
        with api_call("http://127.0.0.1:8000/health", timeout=2) as response:
            print("API sẵn sàng:", json.load(response))
        print("Trọng số/cache model được lưu tại:", MODEL_CACHE)
        break
    except (OSError, urllib.error.URLError):
        if attempt % 15 == 0:
            print("Đang tải/nạp model...", attempt * 2, "giây; log:", server_log)
        time.sleep(2)
else:
    server_process.terminate()
    try:
        server_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait()
    print(server_log.read_text(errors="replace")[-10000:])
    raise TimeoutError("Chưa sẵn sàng sau khoảng 20 phút; kiểm tra tải model trong log.")


## 6. Thử tạo tiếng Việt ngay trong notebook

Phải nghe được âm thanh trước khi kết nối ReviewTrans. WAV được tạo hoàn chỉnh rồi trả về;
đây không phải API phát âm thanh theo từng token.


In [ ]:
# 6 · Kiểm tra tạo âm thanh

from IPython.display import Audio, display
SELECTED_VOICE = "auto"  # Thêm và chọn giọng clone trên Web UI ở phần 7.
with api_call("http://127.0.0.1:8000/v1/audio/speech", {
    "model": "omnivoice", "input": "Xin chào, đây là giọng đọc thử từ OmniVoice trên Google Colab.",
    "voice": SELECTED_VOICE, "response_format": "wav", "speed": 1.0,
}) as response:
    wav_data = response.read()
assert wav_data[:4] == b"RIFF", "API không trả WAV hợp lệ"
(APP_DIR / "test.wav").write_bytes(wav_data)
display(Audio(wav_data))
print("Đã thử suy luận thật trên GPU trong phiên Colab này.")


## 7. Mở Web UI và lấy cấu hình ReviewTrans

Chỉ chạy sau khi ô 6 thành công. Một URL HTTPS dùng cho cả Web UI và API.
Bấm liên kết đăng nhập trực tiếp trong 5 phút; mã chỉ dùng một lần, không chứa API key.
Nếu có trang xác nhận ngrok, tiếp tục tới Web UI. Trình duyệt dùng phiên đăng nhập riêng;
ReviewTrans vẫn dùng API key ở phần 2. Cần liên kết mới: chạy `open_webui_session()` trong ô mã mới.
Token ngrok được truyền qua biến môi trường, không đặt trong dòng lệnh hay output.
Không có script giữ phiên sống hoặc tự né giới hạn Colab.


In [ ]:
# 7a · Ghi cấu hình tunnel ngrok đã sửa (config v2)

(APP_DIR / 'tunnel.py').write_text('import os, signal, time\nfrom pathlib import Path\nfrom pyngrok import ngrok, conf\ncfg = conf.PyngrokConfig(\n    auth_token=os.environ["NGROK_AUTHTOKEN"],\n    config_version="2",\n    config_path=str(Path(__file__).with_name("ngrok-v2.yml")),\n)\ndef stop(*args):\n    raise SystemExit()\nsignal.signal(signal.SIGTERM, stop)\ntry:\n    tunnel = ngrok.connect(addr="http://127.0.0.1:8000", proto="http", bind_tls=True,\n                           inspect=False, pyngrok_config=cfg)\n    Path(os.environ["TUNNEL_URL_FILE"]).write_text(tunnel.public_url)\n    while True:\n        time.sleep(1)\nexcept Exception as exc:\n    # Show the actual reason without echoing the credential in an exception traceback.\n    detail = str(exc)\n    for value in (os.environ.get("NGROK_AUTHTOKEN", ""), os.environ.get("OMNIVOICE_API_KEY", "")):\n        if value:\n            detail = detail.replace(value, "[HIDDEN]")\n    print("NGROK_START_ERROR:", detail, flush=True)\n    raise SystemExit(1)\nfinally:\n    ngrok.kill(pyngrok_config=cfg)\n', encoding='utf-8')


In [ ]:
# 7b · Mở tunnel và tạo liên kết đăng nhập trực tiếp

def show_tunnel_error():
    import re
    details = tunnel_log.read_text(errors="replace") if tunnel_log.is_file() else "Không có tunnel.log"
    for value in (NGROK_TOKEN, API_KEY):
        if value:
            details = details.replace(value, "[HIDDEN]")
    details = re.sub(r"(?i)(authtoken[\"']?\s*[:=]\s*[\"']?)[^\s\"']+", r"\1[HIDDEN]", details)
    print(details[-12000:])
    if "ERR_NGROK_334" in details:
        print("Domain đang được một phiên khác sử dụng. Dừng đúng tunnel cũ rồi chạy lại ô này.")
    elif "ERR_NGROK_4018" in details:
        print("Ngrok chưa xác thực. Kiểm tra authtoken trong Secrets, rồi chạy lại phần 2 và ô này.")

if server_process.poll() is not None:
    raise RuntimeError("Server đã dừng. Chạy lại ô 5 và 6.")
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    try:
        tunnel_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        tunnel_process.kill()
        tunnel_process.wait()
url_file = APP_DIR / "public-url.txt"
url_file.unlink(missing_ok=True)
tunnel_env = os.environ.copy()
tunnel_env.update({"NGROK_AUTHTOKEN": NGROK_TOKEN, "TUNNEL_URL_FILE": str(url_file)})
tunnel_log = APP_DIR / "tunnel.log"
with tunnel_log.open("w") as log_file:
    tunnel_process = subprocess.Popen([PY, "-u", str(APP_DIR / "tunnel.py")],
        env=tunnel_env, stdout=log_file, stderr=subprocess.STDOUT)
for _ in range(60):
    if tunnel_process.poll() is not None:
        show_tunnel_error()
        raise RuntimeError("Không mở được ngrok. Nguyên nhân đã được in ở trên và che giá trị khóa.")
    if url_file.is_file():
        PUBLIC_URL = url_file.read_text().strip().rstrip("/")
        break
    time.sleep(1)
else:
    tunnel_process.terminate()
    show_tunnel_error()
    raise TimeoutError("Ngrok chưa mở sau 60 giây; kiểm tra tunnel.log.")
if not PUBLIC_URL.startswith("https://"):
    raise RuntimeError("Tunnel phải là HTTPS; kiểm tra cấu hình ngrok.")
with api_call(PUBLIC_URL + "/health", timeout=30) as response:
    print("HTTPS đã kiểm tra:", json.load(response))
print("\nWEB UI:", PUBLIC_URL + "/")
def open_webui_session():
    import html
    import json
    from urllib.parse import quote
    from IPython.display import HTML, display

    with api_call("http://127.0.0.1:8000/ui/launch", payload={}, timeout=30) as response:
        launch = json.load(response)
    login_url = PUBLIC_URL.rstrip("/") + "/?ui=2.1#login=" + quote(launch["ticket"], safe="")
    display(HTML('<p><a href="' + html.escape(login_url, quote=True)
                 + '" target="_blank" rel="noopener noreferrer"><b>Mở Web UI · đăng nhập trực tiếp</b></a></p>'))
    print("Bấm liên kết trong 5 phút; mã đăng nhập chỉ dùng một lần, không chứa API key.")
    print("Nếu cần liên kết mới, chạy: open_webui_session()")


open_webui_session()

print("\nĐIỀN VÀO REVIEWTRANS")
print("Tên          : OmniVoice Colab")
print("Loại         : API kiểu OpenAI (/v1/audio/speech)")
print("Base URL     :", PUBLIC_URL + "/v1")
print("API key      : chuỗi bạn nhập ở ô 2; KHÔNG thêm chữ Bearer")
print("Model        : omnivoice")
print("Region       : để trống")
print("Giọng        :", globals().get("SELECTED_VOICE", "auto"))
print("Định dạng    : wav")
print("Số luồng     :", MAX_BATCH_SIZE)
print("Chỉ dẫn giọng : để trống khi dùng cloned; có thể để trống khi thử auto")
print("\nBấm lưu → lấy model/chọn giọng nếu cần → thử giọng → đặt làm mặc định TTS.")

print("Giọng clone mới: thêm trên Web UI, rồi chọn đúng mã giọng trong ReviewTrans.")


## Tra lỗi / dừng phiên (tùy chọn)

- **401:** API key trên PC phải giống ô 2, không thêm `Bearer` trong giao diện ReviewTrans.
- **404:** Base URL phải là `https://.../v1`, không thêm `/audio/speech` vào ô Base URL.
- **429:** hàng đợi đầy hoặc đang thêm giọng. Chờ rồi thử lại.
- **503 / CUDA out of memory:** chia văn bản ngắn hơn, số luồng 1; nếu GPU còn đầy, dừng server và chạy lại ô 5.
- **Hết thời gian 180 giây:** ReviewTrans đang dùng timeout 180 giây cho lệnh speech;
  thử một câu ngắn, `NUM_STEP=16`, rồi khởi động lại server ở ô 5.
- **Không thấy giọng mới trong ReviewTrans:** bấm Làm mới trong Web UI để kiểm tra giọng đã có;
  ReviewTrans có thể cache 10 phút. Khởi động lại ứng dụng để làm mới ngay.
- **Thiết lập không thay đổi API:** bấm Lưu thiết lập cho API trên Web UI. Thay slider chỉ ảnh hưởng
  lần nghe thử cho đến khi lưu. Tốc độ và mã giọng vẫn do ReviewTrans gửi từng yêu cầu.
- **Giọng/thiết lập được lưu ở đâu?** `MyDrive/OmniVoice/voices` và `MyDrive/OmniVoice/settings.json`.
  Kết nối đúng tài khoản Drive ở phiên sau. Thư mục model vẫn là `MyDrive/OmniVoice/models`.
- **Ngrok báo ERR_NGROK / tunnel unavailable:** xem tunnel.log, kiểm tra tài khoản/authtoken,
  endpoint khác đang dùng và quota. Không dán log có khóa lên chat.
- **Colab ngắt:** kết nối GPU lại, chạy lại notebook; lấy Base URL hiện được in ở ô 7.
  ngrok có thể giữ cùng domain tùy tài khoản, nhưng endpoint vẫn cần phiên Colab đang chạy.
- **Model có mất khi Colab ngắt không?** Không, nếu giữ lưu Drive ở phần 1. Kết nối lại
  đúng tài khoản Drive và giữ nguyên đường dẫn để dùng lại file đã tải. Thư viện Python
  cần cài lại trong máy ảo mới. File thiếu, tải dở hoặc thay đổi trên Hub có thể cần tải thêm.
- **Drive báo đầy hoặc lỗi I/O:** kiểm tra dung lượng/tài khoản Drive; xử lý lỗi trước
  khi chạy phần 5. Không xóa thư mục model nếu muốn tiếp tục dùng lại cache.

Ô dưới chỉ in log server. Ô dừng cuối mặc định không chạy lệnh dừng, nên **Run all** không
làm tắt API vừa khởi động.


In [ ]:
# Tùy chọn · Xem log server

print((APP_DIR / "server.log").read_text(errors="replace")[-8000:])


In [ ]:
# Tùy chọn · Dừng khi dùng xong (STOP=False khi Chạy tất cả)

STOP = False  # Đổi thành True và chạy ô này khi đã dùng xong.
if STOP:
    for process_name in ["tunnel_process", "server_process"]:
        process = globals().get(process_name)
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
    print("Đã dừng API/tunnel do notebook này tạo. Có thể ngắt runtime Colab.")
else:
    print("API vẫn chạy. Đổi STOP=True khi muốn dừng.")


## Nguồn và phạm vi kiểm chứng

- [OmniVoice GitHub](https://github.com/k2-fsa/OmniVoice),
  [notebook chính thức](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb).
- [OmniVoice 0.2.1](https://pypi.org/project/omnivoice/0.2.1/) và
  [mã của bản phát hành](https://github.com/k2-fsa/OmniVoice/tree/5ba967c4d5b0f08244ae856b033eea583d1e4517).
- [Model card và giấy phép](https://huggingface.co/k2-fsa/OmniVoice): mã Apache 2.0;
  trọng số CC-BY-NC theo model card. Xem điều kiện trước khi dùng cho mục đích thương mại.
- [Ngrok với Google Colab](https://ngrok.com/docs/using-ngrok-with/googleColab),
  [giới hạn ngrok](https://ngrok.com/docs/pricing-limits/free-plan-limits),
  [giới hạn Colab](https://research.google.com/colaboratory/faq.html).
- [Cấu hình cache Hugging Face](https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables):
  `HF_HUB_CACHE` được trỏ vào Drive trước khi tiến trình server nhập thư viện.

Trước khi bàn giao, adapter được kiểm tra cục bộ bằng FastAPI TestClient với model giả lập
và giao thức của ReviewTrans. Luồng thêm giọng, khôi phục mẫu/thiết lập và Web UI
cũng được kiểm tra cục bộ với model giả lập. Việc tải trọng số, chạy OmniVoice thật trên Colab và mở tunnel
phải được xác nhận khi các ô 3, 5, 6, 7 chạy thành công trong tài khoản của bạn.
Không có khóa thật hoặc giọng mẫu cá nhân được đóng gói trong file này.
